# Loadsmart — Analytics Engineer Challenge

Notebook com funções utilitárias e exportações a partir do DuckDB.

**Pré-requisitos** (já instalados no `.venv`):
```
pip install duckdb pandas paramiko
```

Execute o pipeline antes de rodar este notebook:
```bash
python scripts/ingest.py
cd dbt && dbt run
```

## 1. Setup

In [1]:
import os
from pathlib import Path

import duckdb
import pandas as pd

PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == "notebooks" else Path(os.getcwd())
DUCKDB_PATH = str(PROJECT_ROOT / "data" / "loadsmart.duckdb")

print(f"DuckDB: {DUCKDB_PATH}")
print(f"Arquivo existe: {Path(DUCKDB_PATH).exists()}")

DuckDB: /Users/abner.rodrigues/cases/loadsmart_case/data/loadsmart.duckdb
Arquivo existe: True


## 2. Conexão com DuckDB

In [2]:
con = duckdb.connect(DUCKDB_PATH, read_only=True)

# Confirma tabelas disponíveis
con.execute("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,loadsmart,main_intermediate,int_shipments,"[LOADSMART_ID, LANE_RAW, PICKUP_CITY, PICKUP_S...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False
1,loadsmart,main_intermediate,int_shipments_enriched,"[loadsmart_id, lane_raw, pickup_city, pickup_s...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False
2,loadsmart,main_mart,dim_carrier,"[carrier_sk, carrier_name, carrier_rating, vip...","[VARCHAR, VARCHAR, DOUBLE, BOOLEAN, BIGINT]",False
3,loadsmart,main_mart,dim_date,"[DATE_SK, DATE_DAY, YEAR, QUARTER, MONTH, MONT...","[VARCHAR, TIMESTAMP, INTEGER, INTEGER, INTEGER...",False
4,loadsmart,main_mart,dim_location,"[location_sk, city, state]","[VARCHAR, VARCHAR, VARCHAR]",False
5,loadsmart,main_mart,dim_shipper,"[shipper_sk, SHIPPER_NAME]","[VARCHAR, VARCHAR]",False
6,loadsmart,main_mart,fct_shipments,"[LOADSMART_ID, carrier_sk, shipper_sk, pickup_...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False
7,loadsmart,main_staging,stg_shipments,"[LOADSMART_ID, LANE_RAW, PICKUP_CITY, PICKUP_S...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False
8,loadsmart,raw,shipments,"[loadsmart_id, lane, quote_date, book_date, so...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False


## 3. Função `split_lane()`

Recebe uma string de lane no formato `"City,ST -> City,ST"` e retorna um dicionário com as cidades e estados de origem e destino.

In [3]:
def split_lane(lane: str) -> dict:
    """
    Parseia uma lane no formato 'City,ST -> City,ST'.

    Parâmetros
    ----------
    lane : str
        String de lane, ex: 'Chicago,IL -> Dallas,TX'

    Retorna
    -------
    dict com chaves:
        pickup_city, pickup_state, delivery_city, delivery_state

    Lança
    -----
    ValueError se o formato for inválido.
    """
    if not lane or " -> " not in lane:
        raise ValueError(f"Formato de lane inválido: {lane!r}. Esperado: 'City,ST -> City,ST'")

    origin, destination = lane.split(" -> ", maxsplit=1)

    def _parse_side(side: str) -> tuple[str, str]:
        parts = side.split(",", maxsplit=1)
        if len(parts) != 2:
            raise ValueError(f"Lado da lane malformado: {side!r}")
        return parts[0].strip(), parts[1].strip()

    pickup_city, pickup_state = _parse_side(origin)
    delivery_city, delivery_state = _parse_side(destination)

    return {
        "pickup_city": pickup_city,
        "pickup_state": pickup_state,
        "delivery_city": delivery_city,
        "delivery_state": delivery_state,
    }


# --- Testes manuais ---
cases = [
    ("Chicago,IL -> Dallas,TX",      {"pickup_city": "Chicago",   "pickup_state": "IL", "delivery_city": "Dallas",  "delivery_state": "TX"}),
    ("New York,NY -> Los Angeles,CA", {"pickup_city": "New York",  "pickup_state": "NY", "delivery_city": "Los Angeles", "delivery_state": "CA"}),
    (" Miami , FL ->  Atlanta , GA ", {"pickup_city": "Miami",     "pickup_state": "FL", "delivery_city": "Atlanta", "delivery_state": "GA"}),
]

for lane_str, expected in cases:
    result = split_lane(lane_str)
    assert result == expected, f"FALHA: {lane_str!r} → {result}"
    print(f"OK  {lane_str!r}")
    print(f"    {result}")

OK  'Chicago,IL -> Dallas,TX'
    {'pickup_city': 'Chicago', 'pickup_state': 'IL', 'delivery_city': 'Dallas', 'delivery_state': 'TX'}
OK  'New York,NY -> Los Angeles,CA'
    {'pickup_city': 'New York', 'pickup_state': 'NY', 'delivery_city': 'Los Angeles', 'delivery_state': 'CA'}
OK  ' Miami , FL ->  Atlanta , GA '
    {'pickup_city': 'Miami', 'pickup_state': 'FL', 'delivery_city': 'Atlanta', 'delivery_state': 'GA'}


In [4]:
# Aplicando em amostra real do DuckDB
sample = con.execute("""
    SELECT LANE_RAW
    FROM main_mart.fct_shipments
    LIMIT 5
""").df()

pd.DataFrame(sample["LANE_RAW"].apply(split_lane).tolist())

,pickup_city,pickup_state,delivery_city,delivery_state
0,Cabazon,CA,Tolleson,AZ
1,Hawkins,TX,Roanoke,TX
2,Lodi,CA,Pacific,WA
3,Hawkins,TX,Roanoke,TX
4,Hawkins,TX,Roanoke,TX


In [ ]:
con.close()
print("Conexão DuckDB fechada.")